# LT-Mem 입력 준비 · scene0462_00

공식 LT-Mem 실행이 아닙니다. 공개 코드가 Code (TBD)인 상태에서, 기존 Task 4와 동일한 예측 점군으로 관측 JSON을 생성합니다. 단일 촬영 데이터이며 세션 간 변화·Tri-Memory·VQA는 실행하지 않습니다. CPU 런타임으로 충분합니다. Colab 실행은 아직 검증되지 않았습니다.


In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path('/content/spacewatch3d-lt-mem-input')
if repo.exists():
    raise RuntimeError('Use a fresh runtime or a new checkout directory')
subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch',
                'task/5a-change-detection/lt-mem-dataset-trial',
                'https://github.com/choonpal/spacewatch3d.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)
os.chdir(repo)
print(sys.version)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
import hashlib
from urllib.request import urlretrieve
source = Path('data/lt_mem_trial/scene0462_00.json')
source.parent.mkdir(parents=True, exist_ok=True)
urlretrieve('https://nvlabs.github.io/SpaCeFormer/assets/pointclouds/scene0462_00.json', source)
assert hashlib.sha256(source.read_bytes()).hexdigest() == '5a363f8a444d13827e4113f6f1dad1676b90c9cbbca7af4490d1cdfad21f9abb'


In [ ]:
import json
subprocess.run([sys.executable, '-m', 'spacewatch3d.tasks.task_5a_change_detection.prepare_observations',
                '--source', str(source), '--output', 'outputs/lt_mem_input_trial/observations.json'], check=True)
observations = json.loads(Path('outputs/lt_mem_input_trial/observations.json').read_text())
assert observations['audit']['source_point_count'] == 60000
assert observations['audit']['observation_count'] == 50
assert observations['provenance']['source_matches_task4']
print(json.dumps(observations['audit'], indent=2))


In [ ]:
subprocess.run([sys.executable, '-m', 'spacewatch3d', 'check-fixtures'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/lt_mem_input_trial', 'zip', 'outputs/lt_mem_input_trial')
files.download(archive)


## 후속 입력

공식 LT-Mem 코드가 공개되면 검증한 revision으로 설치하고, 서로 다른 실제 촬영 회차를 공통 좌표계로 정합한 객체 관측과 visual embedding을 연결해야 합니다. 그 뒤 메모리를 생성하고 VQA가 조회합니다. 이 노트북의 색상 히스토그램을 논문 embedding과 동일하다고 해석하지 마세요.
